# Model 2: Real Estate Investment ROI Analysis

Objective: Compare cash purchases and mortgage-financed purchases under a $10 million investment budget.

Metrics:
- Net Operating Income (NOI)
- Annual Cash Flow
- Cash-on-Cash Return
- Return on Investment (ROI)

# Import Libraries

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np


# Set global variables

In [2]:
TOTAL_BUDGET = 10_000_000

## Assumptions

In [3]:
DOWN_PAYMENT_RATE = 0.25
ANNUAL_INTEREST_RATE = 0.05
AMORTIZATION_YEARS = 25

# Mortgage Calculator

In [4]:
def monthly_mortgage_payment(property_price, down_payment_rate=0.25, annual_interest_rate=0.05, amortization_years=25):
    
    loan_amount = property_price * (1 - down_payment_rate)

    monthly_rate = annual_interest_rate / 12

    number_of_payments = amortization_years * 12

    if loan_amount == 0:
        return 0.0

    if monthly_rate == 0:
        return loan_amount / number_of_payments

    payment = loan_amount * monthly_rate * (1 + monthly_rate) ** number_of_payments/ ((1 + monthly_rate) ** number_of_payments - 1)

    return payment

# Investment Return Calculator

In [5]:
def calculate_investment(property_price, annual_revenue, annual_operating_expenses, down_payment_rate=0.25, annual_interest_rate=0.05, amortization_years=25):
    
    down_payment = property_price * down_payment_rate
    
    loan_amount = property_price - down_payment
    
    # how much we need to pay per month
    monthly_payment = monthly_mortgage_payment(property_price, down_payment_rate, annual_interest_rate, amortization_years)

    # how much we need to pay per year
    annual_mortgage_payment = monthly_payment * 12

    # how much money we will earn
    net_operating_income = annual_revenue - annual_operating_expenses
    annual_cash_flow = net_operating_income - annual_mortgage_payment

    # OMG ITS COC!!!!!
    cash_on_cash_return = (
        annual_cash_flow / down_payment
        if down_payment > 0
        else np.nan
    )

    
    cap_rate = (
        net_operating_income / property_price
        if property_price > 0
        else np.nan
    )

    return {
        "property_price": property_price,
        "annual_revenue": annual_revenue,
        "annual_operating_expenses": annual_operating_expenses,
        "net_operating_income": net_operating_income,
        "loan_amount": loan_amount,
        "down_payment": down_payment,
        "annual_mortgage_payment": annual_mortgage_payment,
        "annual_cash_flow": annual_cash_flow,
        "cash_on_cash_return": cash_on_cash_return,
        "cap_rate": cap_rate
    }


## Example

In [6]:
example = calculate_investment(
    property_price=1_000_000,
    annual_revenue=60_000,
    annual_operating_expenses=20_000,
    down_payment_rate=0.25
)

example_df = pd.DataFrame([example])

print(example_df.T.round(2))

                                    0
property_price             1000000.00
annual_revenue               60000.00
annual_operating_expenses    20000.00
net_operating_income         40000.00
loan_amount                 750000.00
down_payment                250000.00
annual_mortgage_payment      52613.10
annual_cash_flow            -12613.10
cash_on_cash_return             -0.05
cap_rate                         0.04


# Compare Cash Purchase vs Mortgage

In [7]:
cash_purchase = calculate_investment(
    property_price=1_000_000,
    annual_revenue=60_000,
    annual_operating_expenses=20_000,
    down_payment_rate=1.0
)

mortgage_purchase = calculate_investment(
    property_price=1_000_000,
    annual_revenue=60_000,
    annual_operating_expenses=20_000,
    down_payment_rate=0.25
)

financing_comparison = pd.DataFrame([
    {"financing": "Cash", **cash_purchase},
    {"financing": "Mortgage (25% Down)", **mortgage_purchase}
])

print(
    financing_comparison[
        [
            "financing",
            "down_payment",
            "loan_amount",
            "net_operating_income",
            "annual_mortgage_payment",
            "annual_cash_flow",
            "cash_on_cash_return",
            "cap_rate"
        ]
    ].round(3).to_string(index=False)
)


          financing  down_payment  loan_amount  net_operating_income  annual_mortgage_payment  annual_cash_flow  cash_on_cash_return  cap_rate
               Cash     1000000.0          0.0                 40000                    0.000         40000.000                 0.04      0.04
Mortgage (25% Down)      250000.0     750000.0                 40000                52613.104        -12613.104                -0.05      0.04


## Neighbourhood and Property Type Analysis

In [9]:

# Load all historical Airbnb listings
airbnb = pd.read_csv(
    "../../data/output/listings_cleaned.csv.gz"
)

# Keep records with usable revenue and prices
market = airbnb.loc[
    airbnb["estimated_revenue_l365d"].notna()
    & airbnb["price"].gt(0)
].copy()

# Group by neighbourhood AND property category
property_market = (
    market
    .groupby(
        ["neighbourhood", "property_category"]
    )
    .agg(
        listings=("listing_id", "count"),
        median_nightly_price=("price", "median"),
        median_occupied_nights=(
            "estimated_occupancy_l365d", "median"
        ),
        median_annual_revenue=(
            "estimated_revenue_l365d", "median"
        ),
        mean_annual_revenue=(
            "estimated_revenue_l365d", "mean"
        )
    )
    .reset_index()
)

# Flag small samples rather than silently dropping them
property_market["sample_size_flag"] = (
    property_market["listings"].apply(
        lambda n: "Small sample" if n < 10 else "OK"
    )
)

# Sort by annual revenue, NOT ROI yet
property_market = property_market.sort_values(
    "median_annual_revenue",
    ascending=False
)

print("\nNEIGHBOURHOOD + PROPERTY TYPE COMPARISON")
print(
    property_market
    .round(2)
    .to_string(index=False)
)

# Export for later purchase-price and ROI calculations
Path("output").mkdir(exist_ok=True)

property_market.to_csv(
    "output/neighbourhood_property_market.csv",
    index=False
)



NEIGHBOURHOOD + PROPERTY TYPE COMPARISON
           neighbourhood                     property_category  listings  median_nightly_price  median_occupied_nights  median_annual_revenue  mean_annual_revenue sample_size_flag
                Oakridge                          Shared Rooms         1               1146.88                   255.0               292454.0            292454.00     Small sample
                West End              Commercial / Hospitality         3                356.00                   246.0                87576.0             79892.00     Small sample
               Killarney Niche / Unconventional Accommodations         1                468.00                   144.0                67392.0             67392.00     Small sample
              Strathcona        Secondary Suites / Guest Units         1                256.20                   255.0                65331.0             65331.00     Small sample
Kensington-Cedar Cottage Niche / Unconventional Accommodat